In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
import numpy as np
from PIL import Image
import os
from tqdm.notebook import tqdm
import random

# Set all seeds for reproducibility
def set_seeds(seed=69):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

CONFIG = {
    'batch_size': 32,
    'num_attributes': 85,
    'num_classes': 50,
    'image_size': 224,
    'seed': 69
}

class AnimalAttributeDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, self.image_paths[idx]
        except Exception as e:
            print(f"Error loading image {self.image_paths[idx]}: {str(e)}")
            return None

def collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if len(batch) == 0:
        raise RuntimeError("Batch is empty after filtering")
    images = torch.stack([item[0] for item in batch])
    paths = [item[1] for item in batch]
    return images, paths

class CALModel(nn.Module):
    def __init__(self, num_attributes, num_classes):
        super(CALModel, self).__init__()
        
        self.backbone = models.resnet50(pretrained=True)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        self.attribute_predictor = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.BatchNorm1d(512),
            nn.Linear(512, num_attributes),
            nn.Sigmoid()
        )
        
        self.class_predictor = nn.Sequential(
            nn.Linear(num_attributes, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.BatchNorm1d(512),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        attributes = self.attribute_predictor(features)
        classes = self.class_predictor(attributes)
        return attributes, classes

def predict():
    set_seeds(CONFIG['seed'])
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    class_df = pd.read_csv('/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/classes.txt', 
                          delimiter='\s+', header=None, names=['index', 'class_name'])
    rev_class_mapping = {idx-1: name for idx, name in zip(class_df['index'], class_df['class_name'])}
    
    test_dir = '/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/test'
    test_paths = sorted([os.path.join(test_dir, f) for f in os.listdir(test_dir) 
                        if f.endswith(('.jpg', '.jpeg', '.png'))])
    
    test_transform = transforms.Compose([
        transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    test_dataset = AnimalAttributeDataset(test_paths, test_transform)
    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    model = CALModel(CONFIG['num_attributes'], CONFIG['num_classes']).to(device)
    checkpoint = torch.load('/kaggle/input/pixelplay2/best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    predictions = []
    
    with torch.no_grad():
        for images, paths in tqdm(test_loader, desc='Predicting'):
            images = images.to(device)
            
            _, outputs = model(images)
            
            _, preds = torch.max(outputs, 1)
            
            for path, pred in zip(paths, preds.cpu().numpy()):
                image_id = os.path.basename(path)
                predicted_class = rev_class_mapping[pred]
                predictions.append({
                    'image_id': image_id,
                    'class': predicted_class
                })
    
    predictions_df = pd.DataFrame(predictions)
    predictions_df = predictions_df.sort_values(by='image_id', ascending=True).reset_index(drop=True)
    
    predictions_df.to_csv('predictions.csv', index=False)
    
    return predictions_df

def verify_predictions_order():
    """Verify that predictions are correctly sorted"""
    df = pd.read_csv('predictions.csv')
    is_sorted = (df['image_id'] == df['image_id'].sort_values()).all()
    print(f"\nVerification: Predictions are {'correctly' if is_sorted else 'not'} sorted by image_id")
    return is_sorted

In [2]:
if __name__ == "__main__":
    predictions_df = predict()
    verify_predictions_order()

Using device: cuda


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 216MB/s]
<ipython-input-1-8a1a14fe74e2>:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is poss

Predicting:   0%|          | 0/94 [00:00<?, ?it/s]


Verification: Predictions are correctly sorted by image_id
